# はじめに

このノートでは、PyTorch Geometricを利用してグラフニューラルネットワークを構築することを目指す。最初はPyTorchの基礎からはじめ、グラフニューラルネットワーク、PyTorch Geometricと内容を進めていく。

今回はグラフニューラルネットワークの数理的な部分をまとめていく。下記のサイトは、グラフニューラルネットワークを理解する上で、非常にわかりやすいので、資料の画像を引用させていただいた。

- [Tutorial 6: Basics of Graph Neural Networks](https://lightning.ai/docs/pytorch/stable/notebooks/course_UvA-DL/06-graph-neural-networks.html)

下記はグラフニューラルネットワークを理解するための参考サイト。


- [グラフニューラルネットワーク | 佐藤 竜馬](https://www.amazon.co.jp/%E3%82%B0%E3%83%A9%E3%83%95%E3%83%8B%E3%83%A5%E3%83%BC%E3%83%A9%E3%83%AB%E3%83%8D%E3%83%83%E3%83%88%E3%83%AF%E3%83%BC%E3%82%AF-%E6%A9%9F%E6%A2%B0%E5%AD%A6%E7%BF%92%E3%83%97%E3%83%AD%E3%83%95%E3%82%A7%E3%83%83%E3%82%B7%E3%83%A7%E3%83%8A%E3%83%AB%E3%82%B7%E3%83%AA%E3%83%BC%E3%82%BA-%E4%BD%90%E8%97%A4-%E7%AB%9C%E9%A6%AC/dp/4065347823)
- [グラフ深層学習のすゝめ。 - YouTube](https://www.youtube.com/watch?v=7rgXi3Xp6NI)
- [GCN — グラフ道場](https://yuya-s.github.io/GraphDojo/01GCN.html)
- [Tutorial 6: Basics of Graph Neural Networks](https://lightning.ai/docs/pytorch/stable/notebooks/course_UvA-DL/06-graph-neural-networks.html)
- [Static and Dynamic Attention: Implications for Graph Neural Networks](https://medium.com/data-science/static-and-dynamic-attention-implications-for-graph-neural-networks-eda0d9d7b60a)
- [CS224W | Home](https://web.stanford.edu/class/cs224w/index.html)
- [nn.labml.ai/ja/graphs](https://nn.labml.ai/#:~:text=%E2%9C%A8%20Graph%20Neural%20Networks)




In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torch_geometric
from torch_geometric.data import Data
from torch_geometric.nn import GATv2Conv
from torch_geometric.loader import DataLoader

import pytorch_lightning as pl

pl.seed_everything(42)

Seed set to 42


42

## グラフとは

<div align="center"><img src="./graph.svg" width="300"></div>

画像元: [Tutorial 6: Basics of Graph Neural Networks](https://lightning.ai/docs/pytorch/stable/notebooks/course_UvA-DL/06-graph-neural-networks.html)

グラフ$\mathcal{G}$は、頂点(Vertices, node)とエッジ(Edges, link)の集合$\mathcal{G}=(V,E)$から構成される。上記のグラフは、頂点$V=\{1,2,3,4\}$とエッジ$E=\{(1,2),(1,3),(3,4)\}$から構成される。このようなエッジの関係は隣接行列$A$で表現される。

$$
\begin{split}A = \begin{bmatrix}
    0 & 1 & 0 & 0\\
    1 & 0 & 1 & 1\\
    0 & 1 & 0 & 1\\
    0 & 1 & 1 & 0
\end{bmatrix}\end{split}
$$

隣接行列$A$は対称行列($A_{ij}=A_{ji}$)なので、行を見ても列を見ても同じではあるが、エッジが存在していれば1となり、頂点間の関係を表現している。

- 1行目は、頂点1から頂点2へのエッジが存在することを表している。
- 2行目は、頂点2から頂点1,3,4へのエッジが存在することを表している。
- 3行目は、頂点3から頂点2,4へのエッジが存在することを表している。
- 4行目は、頂点4から頂点2,3へのエッジが存在することを表している。


PyTorch Geometricでは、エッジリストで隣接関係を表しており、下記のようになる。

```python
# 本当は0-indexで表現するが、画像との対応をわかりやすくするため1-indexで表現している
# --edge_index
[
  [1, 2, 2, 2, 3, 3, 4, 4],
  [2, 1, 3, 4, 2, 4, 2, 3]
]
```

この後の説明に必要な次数行列$D$を説明する。次数行列$D$は、対角行列であり、$D_{ii} = \sum_{j=0} A_{ij}$である。要するにその頂点がどれくらいの頂点と繋がっているのかを表す。

$$
\begin{split}D = \begin{bmatrix}
    1 & 0 & 0 & 0\\
    0 & 3 & 0 & 0\\
    0 & 0 & 2 & 0\\
    0 & 0 & 0 & 2
\end{bmatrix}\end{split}
$$

自己ループを追加した次数行列は

$$
\begin{split}\hat{D} = \begin{bmatrix}
    2 & 0 & 0 & 0\\
    0 & 4 & 0 & 0\\
    0 & 0 & 3 & 0\\
    0 & 0 & 0 & 3
\end{bmatrix}\end{split}
$$

であり、次数の大きい頂点が情報を独占するのを防ぐために、$\hat{D}^{-\frac{1}{2}}$をかける。

$$
\begin{split}\hat{D}^{-\frac{1}{2}} = \begin{bmatrix}
    \frac{1}{\sqrt{2}} & 0 & 0 & 0\\
    0 & \frac{1}{\sqrt{4}}=\frac{1}{2} & 0 & 0\\
    0 & 0 & \frac{1}{\sqrt{3}} & 0\\
    0 & 0 & 0 & \frac{1}{\sqrt{3}}
\end{bmatrix}\end{split}
$$

## Graph Newral Network

すごく簡単な例でグラフニューラルネットワークの計算例をおさらいしておく。実際はいろんなレイヤーや活性化関数を利用するので、あくまでの1エポックの流れを理解する目的。

<div align='center'><img src='./gnn.png' width='1200'></div>


## GCN(Graph Convolutional Networks)

画像の畳み込みと似たようなイメージで、グラフニューラルネットワークにも畳み込みが存在する。下記の引用画像がわかりやすい。接続している頂点の属性情報を交換することで「つながり」という関係を表現する。メッセージを互いに伝達するため「メッセージパッシング」とも呼ばれる。グラフニューラルネットワークでは様々なメッセージパッシング方法が存在しており、どのように情報伝達を行うか、それらを何回繰り返すのかによって、様々な表現を可能にする。

<div align="center"><img src="./mp.svg" width="700"></div>

画像元: [Tutorial 6: Basics of Graph Neural Networks](https://lightning.ai/docs/pytorch/stable/notebooks/course_UvA-DL/06-graph-neural-networks.html)



メッセージパッシングを繰り返すことで、頂点の属性情報を更新していく訳ではあるが、更新を繰り返すことで、グラフの属性が類似するという問題もある。下記の図ではメッセージパッシングが2回行われているが、2回目のメッセージパッシングでは、1つ横の頂点のつながりの情報も受信することになる。これを繰り返すと、全く接続のない頂点間においても、いつかは情報を受信することになる。

<div align="center"><img src="./mp2.png" width="700"></div>

画像元: [グラフ深層学習のすゝめ。](https://www.youtube.com/watch?v=7rgXi3Xp6NI)

定式化するためには、まず頂点が受信するすべてのメッセージをどのように結合するかを決定する必要があり、通常は合計、平均を取ることになる。GCNレイヤー$H^{(l)}$は次のように定義される。

$$
H^{(l+1)} = \sigma\left(\hat{D}^{-\frac{1}{2}} \tilde{A} \hat{D}^{-\frac{1}{2}} H^{(l)} W^{(l)}\right)
$$

- $W^{(l)}$は入力特徴をメッセージに変換する重みパラメタ
- 隣接行列$A$に自己ループを追加した$\tilde{A} = A + I$
- 次数行列$\hat{D}$は、$\hat{D}_{ii} = \sum_{j=0} \tilde{A}_{ij}$。次数の大きい頂点が情報を独占するのを防ぐために、$\hat{D}^{-\frac{1}{2}}$をかける。
- $\sigma$は任意の活性化関数を表す。必ずしもシグモイドである必要はない

<div align="center"><img src="./gcn.png" width="700"></div>

画像元: [グラフ深層学習のすゝめ。](https://www.youtube.com/watch?v=7rgXi3Xp6NI)

ここでは正規化を簡略したバージョン(KipfのGCNとは異なる)でGCNの計算イメージを掴んでおく。

In [22]:
class GCNLayer(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.projection = nn.Linear(c_in, c_out)

    def forward(self, node_feats, adj_matrix):
        print(f"node_feats: {node_feats}")
        print(f"adj_matrix: {adj_matrix}")
        num_neighbours = adj_matrix.sum(dim=-1, keepdims=True)
        print(f"num_neighbours: {num_neighbours}")
        node_feats = self.projection(node_feats)
        node_feats = torch.bmm(adj_matrix, node_feats)
        print(f"node_feats: {node_feats}")
        node_feats = node_feats / num_neighbours
        print(f"node_feats: {node_feats}")
        return node_feats

In [23]:
node_feats = torch.arange(8, dtype=torch.float32).view(1, 4, 2)
adj_matrix = torch.Tensor([[[1, 1, 0, 0], [1, 1, 1, 1], [0, 1, 1, 1], [0, 1, 1, 1]]])
layer = GCNLayer(c_in=2, c_out=2)
layer.projection.weight.data = torch.Tensor([[1.0, 0.0], [0.0, 1.0]])
layer.projection.bias.data = torch.Tensor([0.0, 0.0])

with torch.no_grad():
    out_feats = layer(node_feats, adj_matrix)

node_feats: tensor([[[0., 1.],
         [2., 3.],
         [4., 5.],
         [6., 7.]]])
adj_matrix: tensor([[[1., 1., 0., 0.],
         [1., 1., 1., 1.],
         [0., 1., 1., 1.],
         [0., 1., 1., 1.]]])
num_neighbours: tensor([[[2.],
         [4.],
         [3.],
         [3.]]])
node_feats: tensor([[[ 2.,  4.],
         [12., 16.],
         [12., 15.],
         [12., 15.]]])
node_feats: tensor([[[1., 2.],
         [3., 4.],
         [4., 5.],
         [4., 5.]]])


最初の頂点の出力は、自分と接続している2番目の頂点の平均となっている。他のすべての頂点についても同様である。GNNでは、隣接頂点以外の頂点からも特徴量の交換を可能にする必要があり、これは複数のGCN層を適用することで実現でき、一連のGCN層とReLUなどの活性化関数によって構築されうことで様々な表現を行う。

ただ、出力結果を見るとわかるが、問題点がある。頂点3と4は、接続先が同じであるため、出力特徴が同じになる問題がある。そのため、GCNレイヤーは、すべてのメッセージの平均を取るだけでは、ネットワークに頂点固有の情報がなくなる可能性がある。一般的なアプローチは、自己接続の重みを高くする、自己接続に別の重み行列を定義する、アテンションを使用するなどで対処する。

下記は手書きで計算過程のイメージを掴むためのおまけ。ただ、画像は正規化するのを忘れている。

<div align="center"><img src="./wgcn.png" width="500"></div>

In [24]:
node_feats = torch.Tensor([[
  [1, 0],
  [0, 1],
  [1, 1]
  ]]) # fmt: skip
adj_matrix = torch.Tensor([[
  [1, 1, 0],
  [1, 1, 1],
  [0, 1, 1],
  ]]) # fmt: skip
layer = GCNLayer(c_in=2, c_out=2)
# PyTorchではXW^tなので画像に合うように修正
layer.projection.weight.data = torch.Tensor([[1, 3], [2, 4]])
layer.projection.bias.data = torch.Tensor([0.0, 0.0])

with torch.no_grad():
    layer(node_feats, adj_matrix)

node_feats: tensor([[[1., 0.],
         [0., 1.],
         [1., 1.]]])
adj_matrix: tensor([[[1., 1., 0.],
         [1., 1., 1.],
         [0., 1., 1.]]])
num_neighbours: tensor([[[2.],
         [3.],
         [2.]]])
node_feats: tensor([[[ 4.,  6.],
         [ 8., 12.],
         [ 7., 10.]]])
node_feats: tensor([[[2.0000, 3.0000],
         [2.6667, 4.0000],
         [3.5000, 5.0000]]])


## GAT(Graph ATtention Networks)

Attention層は、線形層、重み行列を使用して各頂点にメッセージを作成するが、周囲の頂点特徴量の重要度であるアテンションを学習することで、隣接している頂点の情報を一様集約ではなく、重みづけた集約を可能とする。下記の画像がわかりやすい。GATの計算グラフでは、頂点Cの情報は低く、頂点Bの情報を大きく採用している。

<div align="center"><img src="./gat.png" width="700"></div>

画像元: [グラフ深層学習のすゝめ。](https://www.youtube.com/watch?v=7rgXi3Xp6NI)

グラフニューラルネットのアテンションの説明では下記の図がよく利用されている。$h_{i}, h_{j}$は頂点$i$と$j$の特徴量である。$\mathbf{W}$は重み行列、$\mathbf{a}$はアテンションの重み。$\alpha_{ij}$は頂点$i$と$j$のアテンションの重みである。$||$は連結を表し、$\mathbf{a}\left[\mathbf{W}h_i||\mathbf{W}h_j\right]$は連結した特徴量を$\mathbf{a}$で重みづけたものである。$\mathcal{N}_i$は頂点$i$の隣接頂点の集合である。

<div align="center"><img src="./att.svg" width="300"></div>

画像元: [Tutorial 6: Basics of Graph Neural Networks](https://lightning.ai/docs/pytorch/stable/notebooks/course_UvA-DL/06-graph-neural-networks.html)

$$
\alpha_{ij} = \frac{\exp\left(\text{LeakyReLU}\left(\mathbf{a}\left[\mathbf{W}h_i||\mathbf{W}h_j\right]\right)\right)}{\sum_{k\in\mathcal{N}_i} \exp\left(\text{LeakyReLU}\left(\mathbf{a}\left[\mathbf{W}h_i||\mathbf{W}h_k\right]\right)\right)}
$$

非線形性を一旦取り除き、式を簡略化すると下記のようになる。非線形性がないと、$h_{i}$は分子分母で打ち消しあい、同じ近傍を持つ頂点に対して同じ出力を生成するという、GCNと同じ問題が生じる。ただ、LeakyReLUが頂点にいくらかの依存性を加えることでこの問題は解消される。



$$
\begin{split}\begin{split}
    \alpha_{ij} & = \frac{\exp\left(\mathbf{a}\left[\mathbf{W}h_i||\mathbf{W}h_j\right]\right)}{\sum_{k\in\mathcal{N}_i} \exp\left(\mathbf{a}\left[\mathbf{W}h_i||\mathbf{W}h_k\right]\right)}\\[5pt]
    & = \frac{\exp\left(\mathbf{a}_{:,:d/2}\mathbf{W}h_i+\mathbf{a}_{:,d/2:}\mathbf{W}h_j\right)}{\sum_{k\in\mathcal{N}_i} \exp\left(\mathbf{a}_{:,:d/2}\mathbf{W}h_i+\mathbf{a}_{:,d/2:}\mathbf{W}h_k\right)}\\[5pt]
    & = \frac{\exp\left(\mathbf{a}_{:,:d/2}\mathbf{W}h_i\right)\cdot\exp\left(\mathbf{a}_{:,d/2:}\mathbf{W}h_j\right)}{\sum_{k\in\mathcal{N}_i} \exp\left(\mathbf{a}_{:,:d/2}\mathbf{W}h_i\right)\cdot\exp\left(\mathbf{a}_{:,d/2:}\mathbf{W}h_k\right)}\\[5pt]
    & = \frac{\exp\left(\mathbf{a}_{:,d/2:}\mathbf{W}h_j\right)}{\sum_{k\in\mathcal{N}_i} \exp\left(\mathbf{a}_{:,d/2:}\mathbf{W}h_k\right)}\\
\end{split}\end{split}
$$

すべてのアテンション係数を計算した後に、加重平均を実行して各頂点の出力特徴を計算することでアテンションが計算される。

$$
h_i'=\sigma\left(\sum_{j\in\mathcal{N}_i}\alpha_{ij}\mathbf{W}h_j\right)
$$



In [25]:
class SimpleGAT(nn.Module):
    def __init__(self, c_in, c_out, num_heads, W, b, a, alpha=0.2):
        super().__init__()

        self.H = num_heads
        self.c_out = c_out

        # W: [H*c_out, c_in]
        self.W = nn.Linear(c_in, c_out * num_heads)
        self.W.weight = nn.Parameter(W.clone())
        self.W.bias = nn.Parameter(b.clone())

        # a: [H, 2*c_out]
        self.a = nn.Parameter(a.clone())

        self.leakyrelu = nn.LeakyReLU(alpha)

    def forward(self, h, adj):

        N = h.size(0)
        print("N:", N)
        # -----------------------------
        # 1) z_i = W h_i
        # -----------------------------
        z = self.W(input=h)  # [N, H*c_out]
        print("z:", z)
        z = z.view(N, self.H, self.c_out)
        print("z(after view):\n", z)

        outputs = []

        # -----------------------------
        # 2) 各ヘッドごとに処理
        # -----------------------------
        print("H:", self.H)
        for head in range(self.H):
            print("head:", head)
            # マルチヘッドのテンソルzから「いま処理しているヘッド」の部分だけを抽出
            z_h = z[:, head, :]  # [N, c_out]
            print("z_h:\n", z_h)
            # a を左右分割
            a_left = self.a[head, : self.c_out]
            a_right = self.a[head, self.c_out :]
            print("a_left:", a_left)
            print("a_right:", a_right)
            # Wh1, Wh2
            Wh1 = torch.matmul(z_h, a_left)  # [N]
            Wh2 = torch.matmul(z_h, a_right)  # [N]
            print("Wh1:", Wh1)
            print("Wh2:", Wh2)
            # e_ij = Wh1_i + Wh2_j
            e = Wh1.unsqueeze(1) + Wh2.unsqueeze(
                0
            )  # [N, 1] + [1, N] = [N,N] ブロードキャストで和
            print("e(before unsqueeze):\n", e)
            e = self.leakyrelu(e)
            print("e(after leakyrelu):\n", e)
            # mask
            e = e.masked_fill(adj == 0, float("-inf"))
            print("e(after masked_fill):\n", e)
            # softmax over j
            alpha = F.softmax(e, dim=1)  # [N,N]
            print("alpha(after softmax):\n", alpha)
            # weighted sum
            h_prime = torch.matmul(alpha, z_h)  # [N,c_out]
            print("h_prime:\n", h_prime)
            outputs.append(h_prime)

        # concat heads
        out = torch.cat(outputs, dim=1)  # [N, H*c_out]
        print("out:", out)
        return out

In [26]:
node_feats = torch.arange(8, dtype=torch.float32).view(4, 2)
adj_matrix = torch.tensor(
    [
        [1, 1, 0, 0],
        [1, 1, 1, 1],
        [0, 1, 1, 1],
        [0, 1, 1, 1],
    ],
    dtype=torch.float32,
)
w = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
b = torch.tensor([0.0, 0.0])
a = torch.tensor([[-0.2, 0.3], [0.1, -0.1]])

model = SimpleGAT(
    c_in=2,
    c_out=1,
    num_heads=2,
    W=w,
    b=b,
    a=a,
)

with torch.no_grad():
    out = model(node_feats, adj_matrix)

N: 4
z: tensor([[0., 1.],
        [2., 3.],
        [4., 5.],
        [6., 7.]])
z(after view):
 tensor([[[0.],
         [1.]],

        [[2.],
         [3.]],

        [[4.],
         [5.]],

        [[6.],
         [7.]]])
H: 2
head: 0
z_h:
 tensor([[0.],
        [2.],
        [4.],
        [6.]])
a_left: tensor([-0.2000], requires_grad=True)
a_right: tensor([0.3000], requires_grad=True)
Wh1: tensor([-0.0000, -0.4000, -0.8000, -1.2000])
Wh2: tensor([0.0000, 0.6000, 1.2000, 1.8000])
e(before unsqueeze):
 tensor([[ 0.0000,  0.6000,  1.2000,  1.8000],
        [-0.4000,  0.2000,  0.8000,  1.4000],
        [-0.8000, -0.2000,  0.4000,  1.0000],
        [-1.2000, -0.6000,  0.0000,  0.6000]])
e(after leakyrelu):
 tensor([[ 0.0000,  0.6000,  1.2000,  1.8000],
        [-0.0800,  0.2000,  0.8000,  1.4000],
        [-0.1600, -0.0400,  0.4000,  1.0000],
        [-0.2400, -0.1200,  0.0000,  0.6000]])
e(after masked_fill):
 tensor([[ 0.0000,  0.6000,    -inf,    -inf],
        [-0.0800,  0.2000,  0

下記の動画はわかりやすいのでおすすめ。

- [Understanding Graph Attention Networks - YouTube](https://www.youtube.com/watch?v=A-yKQamf2Fc&list=PLV8yxwGOxvvoNkzPfCx2i8an--Tkt7O8Z&index=6)

また、下記の画像は手書きで計算過程のイメージを掴むためのおまけ。

<div align="center"><img src="./wgat.png" width="1000"></div>


## GATv2(Graph ATtention v2 Networks)

GATが静的なアテンションだったのに対し、動的なアテンションに変更したものがGATv2である。

- [Static and Dynamic Attention: Implications for Graph Neural Networks](https://medium.com/data-science/static-and-dynamic-attention-implications-for-graph-neural-networks-eda0d9d7b60a)

GATとGATv2はどちらも重み付き近傍集約にアテンションを活用するが、アテンションスコアの計算方法を変えることで、隣接頂点間の関係性の表現力をあげている(必ず優れているわけではない)。計算方法が大きく異なるのではなく、順序を変えることで表現を変更している。GATではアテンションを反映してから活性化関数で変換する。

$$
e_{i,j} = \sigma(\alpha^{T} [W_{l}h_{i} || W_{r}h_{j}])
$$

一方で、GATv2ではアテンションを反映してから活性化関数で変換する。

$$
e_{i,j} = \alpha^{T} \sigma([W_{l}h_{i} + W_{r}h_{j}])
$$

In [27]:
class SimpleGATv2(nn.Module):
    def __init__(self, c_in, c_out, num_heads, W_l, W_r, b_l, b_r, a, alpha=0.2):
        super().__init__()

        self.H = num_heads
        self.c_out = c_out
        self.W_l = nn.Linear(c_in, c_out * num_heads)
        self.W_r = nn.Linear(c_in, c_out * num_heads)
        self.W_l.weight = nn.Parameter(W_l.clone())
        self.W_l.bias = nn.Parameter(b_l.clone())
        self.W_r.weight = nn.Parameter(W_r.clone())
        self.W_r.bias = nn.Parameter(b_r.clone())
        self.a = nn.Parameter(a.clone())
        self.leakyrelu = nn.LeakyReLU(alpha)

    def forward(self, h, adj):
        N = h.size(0)
        print("N:", N)
        # =====================================================
        # 1) Linear projection
        #    z_i^(l) = W_l h_i
        #    z_j^(r) = W_r h_j
        # =====================================================
        z_l = self.W_l(input=h).view(N, self.H, self.c_out)  # [N, H, c_out]
        z_r = self.W_r(input=h).view(N, self.H, self.c_out)  # [N, H, c_out]
        print("z_l:", z_l)
        print("z_r:", z_r)
        outputs = []

        # =====================================================
        # 2) Process each attention head
        # =====================================================
        print("H:", self.H)
        for head in range(self.H):
            print("head:", head)

            # z_i^(l), z_j^(r)
            z_lh = z_l[:, head, :]  # [N, c_out]
            z_rh = z_r[:, head, :]  # [N, c_out]
            print("z_lh:", z_lh)
            print("z_rh:", z_rh)

            # attention weight vector a
            a_h = self.a[head]  # [c_out]
            print("a_h:", a_h)

            # -------------------------------------------------
            # 3) Compute attention scores
            #
            # e_ij = a^T LeakyReLU(z_i^(l) + z_j^(r))
            # -------------------------------------------------
            # z_i + z_j  (broadcast)
            # [N,1,c_out] + [1,N,c_out] -> [N,N,c_out]
            combined = z_lh.unsqueeze(1) + z_rh.unsqueeze(0)
            print("combined:", combined)

            # σ(·) = LeakyReLU
            e = self.leakyrelu(combined)  # [N,N,c_out]
            print("e(after leakyrelu):", e)

            # a^T * ...
            # inner product over c_out dimension
            e = (e * a_h).sum(dim=-1)  # [N,N]
            print("e(after inner product):", e)

            # -------------------------------------------------
            # 4) Mask invalid edges
            # -------------------------------------------------
            e = e.masked_fill(adj == 0, float("-inf"))
            print("e(after masked_fill):", e)

            # -------------------------------------------------
            # 5) Normalize
            #
            # α_ij = softmax_j(e_ij)
            # -------------------------------------------------
            alpha = F.softmax(e, dim=1)  # row-wise normalization
            print("alpha(after softmax):", alpha)

            # -------------------------------------------------
            # 6) Aggregate
            #
            # h'_i = Σ_j α_ij z_j^(r)
            # -------------------------------------------------
            h_prime = torch.matmul(alpha, z_rh)  # [N,c_out]
            print("h_prime:", h_prime)

            outputs.append(h_prime)

        # =====================================================
        # 7) Concatenate heads
        # =====================================================
        # Multi-head concat
        out = torch.cat(outputs, dim=1)
        return out

In [28]:
# 頂点特徴 [N=4, c_in=2]
node_feats = torch.arange(8, dtype=torch.float32).view(4, 2)

# 隣接行列（自己ループあり）
adj_matrix = torch.tensor(
    [
        [1, 1, 0, 0],
        [1, 1, 1, 1],
        [0, 1, 1, 1],
        [0, 1, 1, 1],
    ],
    dtype=torch.float32,
)

# --- モデルパラメータ ---
# multi-head = 2
# c_out = 1 → 各 head の出力次元は1

# W_l, W_r : [H*c_out, c_in] = [2,2]
W_l = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
W_r = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
b_l = torch.tensor([0.0, 0.0])
b_r = torch.tensor([0.0, 0.0])

# a : [H, c_out] = [2,1]
a = torch.tensor([[1.0], [1.0]])

model = SimpleGATv2(
    c_in=2,
    c_out=1,
    num_heads=2,
    W_l=W_l,
    W_r=W_r,
    b_l=b_l,
    b_r=b_r,
    a=a,
)

with torch.no_grad():
    out = model(node_feats, adj_matrix)

print(f"Output:{out}")

N: 4
z_l: tensor([[[0.],
         [1.]],

        [[2.],
         [3.]],

        [[4.],
         [5.]],

        [[6.],
         [7.]]])
z_r: tensor([[[0.],
         [1.]],

        [[2.],
         [3.]],

        [[4.],
         [5.]],

        [[6.],
         [7.]]])
H: 2
head: 0
z_lh: tensor([[0.],
        [2.],
        [4.],
        [6.]])
z_rh: tensor([[0.],
        [2.],
        [4.],
        [6.]])
a_h: tensor([1.], requires_grad=True)
combined: tensor([[[ 0.],
         [ 2.],
         [ 4.],
         [ 6.]],

        [[ 2.],
         [ 4.],
         [ 6.],
         [ 8.]],

        [[ 4.],
         [ 6.],
         [ 8.],
         [10.]],

        [[ 6.],
         [ 8.],
         [10.],
         [12.]]])
e(after leakyrelu): tensor([[[ 0.],
         [ 2.],
         [ 4.],
         [ 6.]],

        [[ 2.],
         [ 4.],
         [ 6.],
         [ 8.]],

        [[ 4.],
         [ 6.],
         [ 8.],
         [10.]],

        [[ 6.],
         [ 8.],
         [10.],
         [12

検算のためにPyTorch GeometricのGATv2Convレイヤーを使用して、計算結果を確認する。

In [29]:
# 1. データの準備 (PyG形式)
# node_feats: [4, 2]
node_feats = torch.arange(8, dtype=torch.float32).view(4, 2)

# edge_index: [2, E] (隣接行列をエッジリスト形式に変換)
# adj_matrixで1の部分のインデックスを取得
adj_matrix = torch.tensor(
    [
        [1, 1, 0, 0],
        [1, 1, 1, 1],
        [0, 1, 1, 1],
        [0, 1, 1, 1],
    ],
    dtype=torch.float32,
)

edge_index = adj_matrix.nonzero().t().contiguous()

# 2. PyGのGATv2Convレイヤーを作成
# concat=True (デフォルト)
conv = GATv2Conv(in_channels=2, out_channels=1, heads=2, bias=True, share_weights=False)

# 3. 重みの移植 (SimpleGATv2で使用した値をセット)
# PyGの内部パラメータ名に合わせて値を流し込む
with torch.no_grad():
    # lin_l, lin_r は [H*c_out, c_in]
    conv.lin_l.weight.copy_(W_l)
    conv.lin_r.weight.copy_(W_r)
    conv.lin_l.bias.copy_(b_l)
    conv.lin_r.bias.copy_(b_r)
    # att は [1, H, c_out] の形状にする必要がある
    # 元の a : [H, c_out] -> [1, 2, 1]
    conv.att.copy_(a.unsqueeze(0))

# 4. 実行
conv.eval()
out_pyg = conv(node_feats, edge_index)

print("PyTorch Geometric Output:")
print(out_pyg)

PyTorch Geometric Output:
tensor([[1.7616, 2.7616],
        [5.6896, 6.6896],
        [5.7019, 6.7019],
        [5.7019, 6.7019]], grad_fn=<AddBackward0>)


また、下記の画像は手書きで計算過程のイメージを掴むためのおまけ。

<div align="center"><img src="./wgatv2_mod.png" width="1000"></div>


GATとGATv2の違いは下記のとおり。[こちら](https://medium.com/data-science/static-and-dynamic-attention-implications-for-graph-neural-networks-eda0d9d7b60a)の記事もわかりやすい。

<div align="center"><img src="./wgatgatv2.png" width="1000"></div>


## ミニバッチ

最後にミニバッチについてまとめておく。大きなデータを効率よく扱う方法として、ミニバッチがある。

バッチ内の各グラフは、頂点とエッジの数が異なる場合があり、単一のテンソルを得るにはパディング処理が必要になる。PyTorch Geometryでは、画像のように連結された頂点とエッジリストを持つ単一のグラフとして利用する。隣接行列は、2つの異なるグラフから来た頂点については0となり、それ以外の場合は個々のグラフの隣接行列によって処理される。これはGraph-level ミニバッチとも呼ばれる。

<div align="center"><img src="./minibatch.png" width="700"></div>

画像元: [Tutorial 6: Basics of Graph Neural Networks](https://lightning.ai/docs/pytorch/stable/notebooks/course_UvA-DL/06-graph-neural-networks.html)



In [30]:
# グラフ1（3頂点）
x1 = torch.tensor([[1.0, 0.0], [2.0, 0.0], [3.0, 0.0]])
edge_index1 = torch.tensor([[0, 1, 2], [1, 2, 0]])
data1 = Data(x=x1, edge_index=edge_index1)

# グラフ2（2頂点）
x2 = torch.tensor([[10.0, 0.0], [20.0, 0.0]])
edge_index2 = torch.tensor([[0], [1]])
data2 = Data(x=x2, edge_index=edge_index2)

# DataLoaderでMini-batch
loader = DataLoader([data1, data2], batch_size=2)

batch = next(iter(loader))
print("Batch:", batch)
print("x:\n", batch.x)
print("edge_index:\n", batch.edge_index)
print("batch_index:\n", batch.batch)

Batch: DataBatch(x=[5, 2], edge_index=[2, 4], batch=[5], ptr=[3])
x:
 tensor([[ 1.,  0.],
        [ 2.,  0.],
        [ 3.,  0.],
        [10.,  0.],
        [20.,  0.]])
edge_index:
 tensor([[0, 1, 2, 3],
        [1, 2, 0, 4]])
batch_index:
 tensor([0, 0, 0, 1, 1])


下記は、出力結果をわかりやすくするために区切り線をいれている。

グラフ2のedge_indexの部分がわかりにくい。本来0-indexedであるが、グラフ1で[0,1,2]を使用しているので、オフセットされて、グラフ2からは3始まりのインデックスで管理される。batchベクトルはどこまでが1つのグラフなのかを管理している。

```python
x:
tensor([[ 1.,  0.],
        [ 2.,  0.],
        [ 3.,  0.],
        -----------
        [10.,  0.],
        [20.,  0.]])
edge_index:
tensor([[0, 1, 2,| 3],
        [1, 2, 0,| 4]])
batch:
tensor([0, 0, 0,| 1, 1])
```

他にも大きなグラフデータに対して、学習効率を高めてスケーリングする方法はいくつかあり、近傍からサンプリングしてグラフを学習する方法などもある。

- [1 つのグラフに対し、PyG の Sampler を利用してMini Batchによる学習を行う](https://www.inoue-kobo.com/ai_ml/torch-geometric-with-sampler/)

また、グラフをクラスター化し、畳み込みを制限しながら、近傍爆発を避けながら学習する方法もある。

- [4. Scaling GNNs.ipynb - Colab](https://colab.research.google.com/drive/1XAjcjRHrSR_ypCk_feIWFbcBKyT4Lirs?usp=sharing)

左側の図のように赤点からスタートし、レイヤーを重ねるごとに近傍が増えていく。一方、右側では、グラフをあらかじめクラスター化することで近傍が爆発しないようにしている。ただ、これでは、元々のつながりが失われてしまうので、クラスター間のエッジを組み込むことで学習を進める。

<div align="center"><img src="./cluster.png" width="700"></div>

画像元: [Cluster-GCN: An Efficient Algorithm for Training Deep and Large Graph Convolutional Networks](https://arxiv.org/abs/1905.07953)

## おまけ

下記のWebアプリでグラフニューラルネットワークがどのように計算されていくのか、イメージを掴める。


- [GNN 101](https://visual-intelligence-umn.github.io/GNN-101/#)

<div align="center"><img src="./img.png" width="700"></div>